数据第一列是节点id，第二列是估计值，上面是post的部分内容，之前将post的原本id映射为0-N的数字，请根据id映射进行表连接，将估计值准确匹配到每个对应的post中

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
merge_sv_with_posts.py
将 Fastest 的无偏节点估计结果 (sv_estimate_result.txt)
与 id_mapping.csv、post.csv 按照内部ID映射合并，
输出每个 Post 的估计值。
"""

import pandas as pd
import os
import re

# =============================
# 🧩 文件路径（请修改为你自己的）
# =============================

BASE_DIR = "/home/wangshuo/projects/FaSTest-main/dataset/sv"

SV_FILE = os.path.join(BASE_DIR, "sv_estimate_result.txt")
MAP_FILE = os.path.join("/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/structure_first", "id_mapping.csv")
POST_FILE = os.path.join("/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/structure_first", "post.csv")

OUTPUT_FILE = os.path.join(BASE_DIR, "post_with_estimate.csv")

# =============================
# 🧩 Step 1. 读取 id 映射
# =============================

idmap = pd.read_csv(MAP_FILE, dtype=str)
idmap["internal_id"] = idmap["internal_id"].astype(int)
posts_map = idmap[idmap["type"] == "Post"][["internal_id", "orig_id"]]
print(f"[INFO] 载入映射表，共 {len(idmap)} 条，Post 类型 {len(posts_map)} 条")

# =============================
# 🧩 Step 2. 解析 sv_estimate_result.txt
# =============================

pattern_line = re.compile(r"^(\d+)\s+([0-9eE\.\-]+)$")
records = []
current_query = None
all_est_value = None

with open(SV_FILE, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith("Query:"):
            current_query = line.split("Query:")[-1].strip()
        elif line.startswith("All Est:"):
            all_est_value = float(line.split(":")[-1].strip())
        elif pattern_line.match(line):
            node_id, val = pattern_line.match(line).groups()
            records.append({
                "internal_id": int(node_id),
                "estimate": float(val),
                "all_est": all_est_value
            })

sv_df = pd.DataFrame(records)
print(f"[INFO] 已读取估计结果: {len(sv_df)} 条节点估计记录")

# =============================
# 🧩 Step 3. 合并 id 映射
# =============================

merged = pd.merge(posts_map, sv_df, on="internal_id", how="inner")
print(f"[INFO] 合并映射后：{len(merged)} 条有效 Post 节点")

# =============================
# 🧩 Step 4. 合并原始 post.csv
# =============================

posts_df = pd.read_csv(POST_FILE, dtype=str)
result = pd.merge(posts_df, merged, left_on="id:ID", right_on="orig_id", how="left")

# =============================
# 🧩 Step 5. 输出结果
# =============================

result.to_csv(OUTPUT_FILE, index=False)
print(f"[✅] 已生成带估计值的 Post 文件: {OUTPUT_FILE}")

# =============================
# 🧩 Step 6. 汇总统计
# =============================

nonzero = result[result["estimate"].astype(float) > 0]
ratio = len(nonzero) / len(result) if len(result) else 0

print("\n===== 📊 统计结果 =====")
print(f"总 Post 数量: {len(result)}")
print(f"估计值非零的 Post 数: {len(nonzero)}")
print(f"非零比率: {ratio:.4%}")
print("估计值范围:")
if not result["estimate"].isna().all():
    print(result["estimate"].astype(float).describe())


[INFO] 载入映射表，共 175688 条，Post 类型 24082 条
[INFO] 已读取估计结果: 22373 条节点估计记录
[INFO] 合并映射后：22373 条有效 Post 节点
[✅] 已生成带估计值的 Post 文件: /home/wangshuo/projects/FaSTest-main/dataset/sv/post_with_estimate.csv

===== 📊 统计结果 =====
总 Post 数量: 24082
估计值非零的 Post 数: 9820
非零比率: 40.7773%
估计值范围:
count    22373.000000
mean         4.928083
std         11.995651
min          0.000000
25%          0.000000
50%          0.000000
75%          5.512800
max        413.460000
Name: estimate, dtype: float64


In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
merge_sv_with_posts.py
将 Fastest 的无偏节点估计结果 (sv_estimate_result.txt)
与 id_mapping.csv、post.csv 按照内部ID映射合并，
输出每个 Post 的估计值。
"""

import pandas as pd
import os
import re

# =============================
# 🧩 文件路径（请修改为你自己的）
# =============================

BASE_DIR = "/home/wangshuo/projects/FaSTest-main/dataset/sv"

SV_FILE = os.path.join(BASE_DIR, "sv_estimate_result.txt")
MAP_FILE = os.path.join("/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/structure_first", "id_mapping.csv")
POST_FILE = os.path.join("/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/structure_first", "post.csv")

OUTPUT_FILE = os.path.join(BASE_DIR, "post_with_estimate.csv")

# =============================
# 🧩 Step 1. 读取 id 映射
# =============================

idmap = pd.read_csv(MAP_FILE, dtype=str)
idmap["internal_id"] = idmap["internal_id"].astype(int)
posts_map = idmap[idmap["type"] == "Post"][["internal_id", "orig_id"]]
print(f"[INFO] 载入映射表，共 {len(idmap)} 条，Post 类型 {len(posts_map)} 条")

# =============================
# 🧩 Step 2. 解析 sv_estimate_result.txt
# =============================

pattern_line = re.compile(r"^(\d+)\s+([0-9eE\.\-]+)$")
records = []
current_query = None
all_est_value = None

with open(SV_FILE, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith("Query:"):
            current_query = line.split("Query:")[-1].strip()
        elif line.startswith("All Est:"):
            try:
                all_est_value = float(line.split(":")[-1].strip())
            except ValueError:
                all_est_value = None
        elif pattern_line.match(line):
            node_id, val = pattern_line.match(line).groups()
            records.append({
                "internal_id": int(node_id),
                "estimate": float(val),
                "all_est": all_est_value
            })

sv_df = pd.DataFrame(records)
print(f"[INFO] 已读取估计结果: {len(sv_df)} 条节点估计记录")

# =============================
# 🧩 Step 3. 合并 id 映射
# =============================

merged = pd.merge(posts_map, sv_df, on="internal_id", how="inner")
print(f"[INFO] 合并映射后：{len(merged)} 条有效 Post 节点")

# =============================
# 🧩 Step 4. 合并原始 post.csv
# =============================

posts_df = pd.read_csv(POST_FILE, dtype=str)
# ⚠️ 部分 CSV 的列名是 "id" 而非 "id:ID"，所以要兼容
id_col = "id:ID" if "id:ID" in posts_df.columns else "id"
result = pd.merge(posts_df, merged, left_on=id_col, right_on="orig_id", how="left")

# =============================
# 🧩 Step 5. 对无法匹配的项设置 estimate = 0
# =============================

result["estimate"] = pd.to_numeric(result["estimate"], errors="coerce").fillna(0.0)

# =============================
# 🧩 Step 6. 输出结果
# =============================

result.to_csv(OUTPUT_FILE, index=False)
print(f"[✅] 已生成带估计值的 Post 文件: {OUTPUT_FILE}")

# =============================
# 🧩 Step 7. 汇总统计
# =============================

nonzero = result[result["estimate"] > 0]
ratio = len(nonzero) / len(result) if len(result) else 0

print("\n===== 📊 统计结果 =====")
print(f"总 Post 数量: {len(result)}")
print(f"估计值非零的 Post 数: {len(nonzero)}")
print(f"非零比率: {ratio:.4%}")
print("估计值范围:")
if not result["estimate"].isna().all():
    print(result["estimate"].describe())


[INFO] 载入映射表，共 175688 条，Post 类型 24082 条
[INFO] 已读取估计结果: 22373 条节点估计记录
[INFO] 合并映射后：22373 条有效 Post 节点
[✅] 已生成带估计值的 Post 文件: /home/wangshuo/projects/FaSTest-main/dataset/sv/post_with_estimate.csv

===== 📊 统计结果 =====
总 Post 数量: 24082
估计值非零的 Post 数: 9820
非零比率: 40.7773%
估计值范围:
count    24082.000000
mean         4.578357
std         11.631198
min          0.000000
25%          0.000000
50%          0.000000
75%          5.512800
max        413.460000
Name: estimate, dtype: float64


将估计值为非0且不空的项保存到OUTPUT_FILE = os.path.join(BASE_DIR, "post_with_estimate.csv")

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
merge_sv_with_posts.py
将 Fastest 的无偏节点估计结果 (sv_estimate_result.txt)
与 id_mapping.csv、post.csv 按照内部ID映射合并，
仅输出估计值非零且不为空的 Post，
并重命名输出列。
"""

import pandas as pd
import os
import re

# =============================
# 🧩 文件路径（请修改为你自己的）
# =============================

BASE_DIR = "/home/wangshuo/projects/FaSTest-main/dataset/sv"

SV_FILE = os.path.join(BASE_DIR, "sv_estimate_result.txt")
MAP_FILE = os.path.join("/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/structure_first", "id_mapping.csv")
POST_FILE = os.path.join("/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/structure_first", "post.csv")

OUTPUT_FILE = os.path.join(BASE_DIR, "post_with_estimate.csv")

# =============================
# 🧩 Step 1. 读取 id 映射
# =============================

idmap = pd.read_csv(MAP_FILE, dtype=str)
idmap["internal_id"] = idmap["internal_id"].astype(int)
posts_map = idmap[idmap["type"] == "Post"][["internal_id", "orig_id"]]
print(f"[INFO] 载入映射表，共 {len(idmap)} 条，Post 类型 {len(posts_map)} 条")

# =============================
# 🧩 Step 2. 解析 sv_estimate_result.txt
# =============================

pattern_line = re.compile(r"^(\d+)\s+([0-9eE\.\-]+)$")
records = []
current_query = None
all_est_value = None

with open(SV_FILE, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith("Query:"):
            current_query = line.split("Query:")[-1].strip()
        elif line.startswith("All Est:"):
            try:
                all_est_value = float(line.split(":")[-1].strip())
            except ValueError:
                all_est_value = None
        elif pattern_line.match(line):
            node_id, val = pattern_line.match(line).groups()
            records.append({
                "internal_id": int(node_id),
                "estimate": float(val),
                "all_est": all_est_value
            })

sv_df = pd.DataFrame(records)
print(f"[INFO] 已读取估计结果: {len(sv_df)} 条节点估计记录")

# =============================
# 🧩 Step 3. 合并 id 映射
# =============================

merged = pd.merge(posts_map, sv_df, on="internal_id", how="inner")
print(f"[INFO] 合并映射后：{len(merged)} 条有效 Post 节点")

# =============================
# 🧩 Step 4. 合并原始 post.csv
# =============================

posts_df = pd.read_csv(POST_FILE, dtype=str)
id_col = "id:ID" if "id:ID" in posts_df.columns else "id"
result = pd.merge(posts_df, merged, left_on=id_col, right_on="orig_id", how="left")

# =============================
# 🧩 Step 5. 对无法匹配的项设置 estimate = 0
# =============================

result["estimate"] = pd.to_numeric(result["estimate"], errors="coerce").fillna(0.0)

# =============================
# 🧩 Step 6. 仅保留估计值非零的项
# =============================

nonzero_result = result[result["estimate"] > 0].copy()

# =============================
# 🧩 Step 7. 重命名输出列
# =============================

rename_map = {
    "ML1_oracle1_probability": "post_oracle1",
    "ML1_proxy4b1_probability": "post_proxy4b1",
    "ML1_proxy2b1_probability": "post_proxy2b1",
    "orig_id": "postId"
}

nonzero_result.rename(columns=rename_map, inplace=True)

# =============================
# 🧩 Step 8. 保存结果
# =============================

nonzero_result.to_csv(OUTPUT_FILE, index=False)
print(f"[✅] 已生成仅包含非零估计值的 Post 文件: {OUTPUT_FILE}")

# =============================
# 🧩 Step 9. 汇总统计
# =============================

total_posts = len(result)
nonzero = len(nonzero_result)
ratio = nonzero / total_posts if total_posts else 0

print("\n===== 📊 统计结果 =====")
print(f"总 Post 数量: {total_posts}")
print(f"估计值非零的 Post 数: {nonzero}")
print(f"非零比率: {ratio:.4%}")
print("估计值范围:")
if not result["estimate"].isna().all():
    print(result["estimate"].describe())


[INFO] 载入映射表，共 175688 条，Post 类型 24082 条
[INFO] 已读取估计结果: 22373 条节点估计记录
[INFO] 合并映射后：22373 条有效 Post 节点
[✅] 已生成仅包含非零估计值的 Post 文件: /home/wangshuo/projects/FaSTest-main/dataset/sv/post_with_estimate.csv

===== 📊 统计结果 =====
总 Post 数量: 24082
估计值非零的 Post 数: 9820
非零比率: 40.7773%
估计值范围:
count    24082.000000
mean         4.578357
std         11.631198
min          0.000000
25%          0.000000
50%          0.000000
75%          5.512800
max        413.460000
Name: estimate, dtype: float64
